In [1]:
import json

In [38]:
with open("supply.json", "r", encoding="utf-8") as file:
    data = json.load(file)
print(data)

{'supply_research': {'search_metadata': {'id': '6a3cc298dc74d02748524c15', 'status': 'Success', 'json_endpoint': 'https://serpapi.com/searches/JppWrBpopX5_U1LuPfsw6Q/6a3cc298dc74d02748524c15.json', 'pixel_position_endpoint': 'https://serpapi.com/searches/JppWrBpopX5_U1LuPfsw6Q/6a3cc298dc74d02748524c15.json_with_pixel_position', 'created_at': '2026-06-25 05:54:32 UTC', 'processed_at': '2026-06-25 05:54:32 UTC', 'google_url': 'https://www.google.com/search?q=portable+blender&oq=portable+blender&uule=w+CAIQICIFSW5kaWE&hl=en&sourceid=chrome&ie=UTF-8', 'raw_html_file': 'https://serpapi.com/searches/JppWrBpopX5_U1LuPfsw6Q/6a3cc298dc74d02748524c15.html', 'total_time_taken': 1.06}, 'search_parameters': {'engine': 'google', 'q': 'portable blender', 'location_requested': 'India', 'location_used': 'India', 'google_domain': 'google.com', 'hl': 'en', 'device': 'desktop'}, 'search_information': {'query_displayed': 'portable blender', 'total_results': 236, 'time_taken_displayed': 0.28, 'organic_resul

In [54]:
def extract_ai_overview_sections(response):

    sections = []
    
    buttons = (
        response
        .get("things_to_know", {})
        .get("buttons", [])
    )  

    for button in buttons:

        section_name = button.get("text", "")

        overview = button.get("ai_overview", {})

        text_blocks = overview.get("text_blocks", [])

        block_text = []

        for block in text_blocks:

            if block.get("snippet"):
                block_text.append(block["snippet"])

            if block.get("type") == "list":

                for item in block.get("list", []):

                    if item.get("title"):
                        block_text.append(item["title"])

                    if item.get("snippet"):
                        block_text.append(item["snippet"])

        sections.append(
            {
                "section": section_name,
                "content": block_text
            }
        )

    return sections

In [59]:
inline_images = [] #[source_name,title]
inline_videos = [] #[title,platform]
related_questions = [] #[question]
organic_results = [] #[title,snippet,about_this_result.source.description , about_this_result.source,source]
related_searches = [] #[query]
immersive_products = []
titles = []
sources = []
questions = []
snippets = []
querys = []
filtered_products = []

supply_research = {}

sections = extract_ai_overview_sections(data["supply_research"])
supply_research["ai_overview"] = sections

if "supply_research" in data:
    supply_data = data["supply_research"]

    if "inline_images" in supply_data:
        #inline_images
        titles  = [item.get("title") for item in supply_data["inline_images"]] 
        sources  = [item.get("source_name") for item in supply_data["inline_images"]] 
        supply_research["inline_images"] = {"titles":titles,"sources":sources}

    if "related_questions" in supply_data:
        questions = [item.get("question") for item in supply_data["related_questions"]] 
        supply_research["questions"] = questions

    if "organic_results" in supply_data:       
        keep_keys = {"snippet", "source", "title"}
        filtered_products = [
            {key: item[key] for key in keep_keys if key in item} 
            for item in supply_data["organic_results"]
        ]
        supply_research["organic_results"] = filtered_products

        #titles = [item.get("title") for item in supply_data["organic_results"]] 
        #snippets = [item.get("snippet") for item in supply_data["organic_results"]] 
        #sources = [item.get("source") for item in supply_data["organic_results"]] 
    
    if "immersive_products" in supply_data:
        keep_keys = {"category", "source", "title","rating","reviews","price","original_price"}
        filtered_products = [
            {key: item[key] for key in keep_keys if key in item} 
            for item in supply_data["immersive_products"]
        ]
        supply_research["immersive_products"] = filtered_products

    if "related_searches" in supply_data:
        querys = [item.get("query") for item in supply_data["related_searches"]]
        supply_research["related_searches"] = querys



In [60]:
supply_research

{'ai_overview': [{'section': 'Price',
   'content': ['Portable blenders typically cost between $25 and $70 for popular, high-quality models, though basic mini options can be found for as low as $12–$15, and premium, high-performance personal blenders can reach over $100. Most USB-rechargeable models for smoothies cost around $30–$50, with popular brands like BlendJet and Ninja sitting in the mid-range.',
    'Price Breakdown by Type:',
    'Budget/Basic Portable ($12 - $25):',
    'Small USB-rechargeable units (e.g., generic brands on Walmart/Amazon), often 14oz-16oz, suited for light smoothies or shakes.',
    'Mid-Range/Popular Portable ($30 - $60):',
    'Popular brands like BlendJet 2 ($30-$40) and Ninja Blast ($45-$70) offer better durability, battery life, and ice-crushing power.',
    'Premium/High-Power Personal ($70 - $150+):',
    'While not always "portable" (rechargeable), high-power personal blenders like Nutribullet or Beast ($90-$130) offer superior blending, though they

In [74]:
def parse_google_trends(serp_data):
    trends_data_processed = []
    trends_data = serp_data["interest_over_time"]
    for entry in trends_data.get("timeline_data"):
        #print(entry)
        # Extract the outer date attribute
        date_val = entry.get("date")
        #print(date_val)
        # Loop through each dictionary inside the "values" array
        # If there are multiple items, this loop will handle all of them for the same date
        for val_item in entry.get("values", []):
            trends_data_processed.append({
                "date": date_val,
                "query": val_item.get("query"),
                "extracted_value": val_item.get("extracted_value")
            })

    return trends_data_processed

In [75]:
with open("trends_data.json", "r", encoding="utf-8") as file:
    data = json.load(file)
    #print(data)
    x = parse_google_trends(data)
    print(x)

[{'date': 'Jun 22\u2009–\u200928, 2025', 'query': 'portable blender', 'extracted_value': 3}, {'date': 'Jun 29\u2009–\u2009Jul 5, 2025', 'query': 'portable blender', 'extracted_value': 3}, {'date': 'Jul 6\u2009–\u200912, 2025', 'query': 'portable blender', 'extracted_value': 4}, {'date': 'Jul 13\u2009–\u200919, 2025', 'query': 'portable blender', 'extracted_value': 4}, {'date': 'Jul 20\u2009–\u200926, 2025', 'query': 'portable blender', 'extracted_value': 4}, {'date': 'Jul 27\u2009–\u2009Aug 2, 2025', 'query': 'portable blender', 'extracted_value': 4}, {'date': 'Aug 3\u2009–\u20099, 2025', 'query': 'portable blender', 'extracted_value': 5}, {'date': 'Aug 10\u2009–\u200916, 2025', 'query': 'portable blender', 'extracted_value': 6}, {'date': 'Aug 17\u2009–\u200923, 2025', 'query': 'portable blender', 'extracted_value': 6}, {'date': 'Aug 24\u2009–\u200930, 2025', 'query': 'portable blender', 'extracted_value': 6}, {'date': 'Aug 31\u2009–\u2009Sep 6, 2025', 'query': 'portable blender', 'ext

In [57]:
with open("postman_demand.json", "r", encoding="utf-8") as file:
    demand = json.load(file)
    print(demand)
    

{'job': {'aggregate_name': None, 'browser_instructions': None, 'callback_url': None, 'client_id': 195245, 'client_notes': None, 'content_encoding': 'utf-8', 'context': [{'key': 'force_headers', 'value': False}, {'key': 'force_cookies', 'value': False}, {'key': 'hc_policy', 'value': True}, {'key': 'category_id', 'value': None}, {'key': 'merchant_id', 'value': None}, {'key': 'check_empty_geo', 'value': None}, {'key': 'safe_search', 'value': True}, {'key': 'currency', 'value': None}, {'key': 'sort_by', 'value': None}, {'key': 'refinements', 'value': None}, {'key': 'min_price', 'value': None}, {'key': 'max_price', 'value': None}, {'key': 'successful_parse_status_codes', 'value': []}], 'created_at': '2026-06-25 18:00:19', 'domain': 'in', 'geo_location': '560078', 'id': '7475971153001978881', 'limit': 10, 'locale': None, 'markdown': False, 'pages': 1, 'parse': True, 'parser_preset': None, 'parser_type': None, 'parsing_instructions': None, 'query': 'portable blender', 'render': None, 'session

In [58]:
refinements = demand.get('results',[])[0].get('content',[]).get('refinements',{})
results = demand.get('results',[])[0].get('content',{}).get('results',{})

In [75]:
results

{'amazons_choices': [{'asin': 'B09J2T124D',
   'best_seller': False,
   'currency': 'INR',
   'is_amazons_choice': True,
   'is_prime': False,
   'is_sponsored': False,
   'manufacturer': '',
   'pos': 3,
   'price': 1599,
   'price_strikethrough': 5000,
   'price_upper': 1599,
   'pricing_count': 1,
   'rating': 4.4,
   'reviews_count': 31700,
   'sales_volume': '9K+ bought in past month',
   'shipping_information': 'FREE delivery Sat, 27 JunOr fastest delivery Tomorrow 8\u202fam - 12\u202fpm',
   'title': 'NutriPro Juicer Mixer Grinder - Smoothie Maker - 500 Watts (2 Jars & 1 Blade, Silver) - 2 Year Warranty',
   'url': '/NutriPro-Bullet-Juicer-Grinder-Blades/dp/B09J2T124D/ref=sr_1_3?dib=eyJ2IjoiMSJ9.1wbE3JTH1AfAxiYO3DjvBh8gmsOPx1433NmXOsg_fDxF1Y1oFia5CVDHyl-jlAzxOLzdyNnVH9FpvcIVtwzRGZs0EwUng3xcJJfu2akgTMc4wBOATIOgCbIS6PBM36uvkxva7NJQbtWgLjg5_To6gFSOKv6KiMIYNf0DdUe9mtbPDyKn1y_TmamA_ncbySMpU3NDeq7O1GWdc0jAt06u1gaO3TouuKW_-wNh1ckyWwU.h6xznIrVyNq0Hw-NCyQfkNZINghAl136ySZ1b33l0AQ&dib_tag=

In [73]:
refinement = {}
result = {}
result_type = []

for rows in refinements:
    row_data = []
    for row in refinements[rows]:
        row_data.append(row["name"])
    refinement[rows] = row_data

for rows in results:
    row_data = {}
    result_type = []
    #print(rows,results[rows])
    for row in results[rows]:
        #print(row)
        row_data = {"asin": row.get("asin"),"price": row.get("price"),"rating": row.get("rating"),
                    "title": row.get("title"),"reviews_count" : row.get("reviews_count")}
        result_type.append(row_data)
    if len(result_type) > 0:
        result[rows] = result_type

In [68]:
refinement

{'availability': ['Include Out of Stock'],
 'brands': ['Ninja',
  'Borosil',
  'KILIG',
  'Pigeon',
  'amazon basics',
  'Ozoy',
  'kuvings',
  'DiCUNO',
  'COOKWELL',
  'BOSS',
  'Hamilton Beach'],
 'capacity': ['Up to 449 mL',
  '450 to 849 mL',
  '850 to 1,499 mL',
  '1,500 mL & above'],
 'colour': ['Apply the filter Black to narrow results',
  'Apply the filter Grey to narrow results',
  'Apply the filter White to narrow results',
  'Apply the filter Brown to narrow results',
  'Apply the filter Beige to narrow results',
  'Apply the filter Red to narrow results',
  'Apply the filter Pink to narrow results',
  'Apply the filter Orange to narrow results',
  'Apply the filter Yellow to narrow results',
  'Apply the filter Green to narrow results',
  'Apply the filter Turquoise to narrow results',
  'Apply the filter Blue to narrow results',
  'Apply the filter Purple to narrow results',
  'Apply the filter Gold to narrow results',
  'Apply the filter Silver to narrow results',
  'App

In [74]:
result

{'amazons_choices': [{'asin': 'B09J2T124D',
   'price': 1599,
   'rating': 4.4,
   'title': 'NutriPro Juicer Mixer Grinder - Smoothie Maker - 500 Watts (2 Jars & 1 Blade, Silver) - 2 Year Warranty',
   'reviews_count': 31700}],
 'organic': [{'asin': 'B0H1Q6LGH1',
   'price': 499,
   'rating': 4.5,
   'title': 'Portable Blender Electric Juicers Fruit Mixers USB Rechargeable Smoothie Mini Personal Juicer 6 Blades 3Gears With 1500 Mah Rechargeable Battery-Smoothie Food Blender D1',
   'reviews_count': 125},
  {'asin': 'B0H21YV2JW',
   'price': 299,
   'rating': 4.2,
   'title': 'Xapiaxa Portable 450 ml Blender 6-Blade – USB Rechargeable Personal Smoothie Maker, Juicer Cup, BPA-Free Travel Mixer (Multicolor)',
   'reviews_count': 14},
  {'asin': 'B09J2T124D',
   'price': 1599,
   'rating': 4.4,
   'title': 'NutriPro Juicer Mixer Grinder - Smoothie Maker - 500 Watts (2 Jars & 1 Blade, Silver) - 2 Year Warranty',
   'reviews_count': 31700},
  {'asin': 'B0F9YGHHXG',
   'price': 698,
   'ratin

In [76]:
import socket

print(socket.gethostbyname("realtime.oxylabs.io"))

gaierror: [Errno 11001] getaddrinfo failed

In [77]:
import httpx

r = httpx.get("https://www.google.com")

print(r.status_code)

302


In [78]:
import socket

hosts = [
    "oxylabs.io",
    "dashboard.oxylabs.io",
    "realtime.oxylabs.io"
]

for host in hosts:
    try:
        print(host, socket.gethostbyname(host))
    except Exception as e:
        print(host, e)

oxylabs.io [Errno 11001] getaddrinfo failed
dashboard.oxylabs.io [Errno 11001] getaddrinfo failed
realtime.oxylabs.io [Errno 11001] getaddrinfo failed


In [79]:
import requests

try:
    r = requests.get("https://realtime.oxylabs.io")
    print(r.status_code)
except Exception as e:
    print(type(e).__name__, e)

ConnectionError HTTPSConnectionPool(host='realtime.oxylabs.io', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000141968D6270>: Failed to resolve 'realtime.oxylabs.io' ([Errno 11001] getaddrinfo failed)"))
